# Validation Readiness Test Suite
Notebook ini memvalidasi kesiapan eksekusi request menggunakan fungsi validasi dan test case terstruktur.

## 1. Setup Environment dan Library

In [ ]:
from typing import Any, Dict, List, Tuple

import pandas as pd
import matplotlib.pyplot as plt

# Konfigurasi tampilan agar tabel dan plot lebih nyaman dibaca.
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 140)
plt.style.use("ggplot")

## 2. Bangun Fungsi Validasi validate_request_readiness()
Fungsi ini menerima input, memvalidasi aturan dasar, lalu mengembalikan tuple `(status_boolean, pesan)`.

In [ ]:
def validate_request_readiness(data: Any) -> Tuple[bool, str]:
    """Validasi sederhana untuk menentukan apakah request siap dieksekusi."""
    if data is None:
        return False, "Input None tidak valid"

    if not isinstance(data, dict):
        return False, "Input harus bertipe dict"

    required_keys = {"task", "confidence", "resources"}
    missing = required_keys - set(data.keys())
    if missing:
        return False, f"Key wajib belum ada: {sorted(missing)}"

    task = data.get("task")
    confidence = data.get("confidence")
    resources = data.get("resources")

    if not isinstance(task, str) or not task.strip():
        return False, "task harus string non-kosong"

    if not isinstance(confidence, (int, float)):
        return False, "confidence harus numerik"

    if confidence < 0 or confidence > 1:
        return False, "confidence harus di rentang 0..1"

    if not isinstance(resources, int):
        return False, "resources harus integer"

    if resources < 1:
        return False, "resources minimal 1"

    if confidence >= 0.7 and resources >= 2:
        return True, "Bisa: input valid dan siap dieksekusi"

    return False, "Belum bisa: confidence atau resources kurang"

## 3. Jalankan Skenario Uji Dasar

In [ ]:
basic_cases: List[Dict[str, Any]] = [
    {
        "name": "basic_valid_high_conf",
        "input": {"task": "labeling", "confidence": 0.9, "resources": 3},
        "expected_status": True,
    },
    {
        "name": "basic_valid_threshold",
        "input": {"task": "training", "confidence": 0.7, "resources": 2},
        "expected_status": True,
    },
    {
        "name": "basic_low_conf",
        "input": {"task": "training", "confidence": 0.6, "resources": 3},
        "expected_status": False,
    },
]

for case in basic_cases:
    actual_status, actual_msg = validate_request_readiness(case["input"])
    print(f"{case['name']}: status={actual_status}, msg={actual_msg}")

## 4. Uji Edge Case dan Penanganan Error

In [ ]:
edge_cases: List[Dict[str, Any]] = [
    {"name": "edge_none", "input": None, "expected_status": False},
    {"name": "edge_empty_dict", "input": {}, "expected_status": False},
    {"name": "edge_wrong_type", "input": [1, 2, 3], "expected_status": False},
    {
        "name": "edge_confidence_extreme_high",
        "input": {"task": "train", "confidence": 1.5, "resources": 2},
        "expected_status": False,
    },
    {
        "name": "edge_confidence_extreme_low",
        "input": {"task": "train", "confidence": -0.1, "resources": 2},
        "expected_status": False,
    },
    {
        "name": "edge_resource_zero",
        "input": {"task": "label", "confidence": 0.9, "resources": 0},
        "expected_status": False,
    },
]

all_cases = basic_cases + edge_cases

def run_case(case: Dict[str, Any]) -> Dict[str, Any]:
    try:
        actual_status, actual_msg = validate_request_readiness(case["input"])
        error_text = ""
    except Exception as exc:
        actual_status = False
        actual_msg = f"Exception: {exc}"
        error_text = repr(exc)

    passed = actual_status == case["expected_status"]
    return {
        "case_name": case["name"],
        "input": case["input"],
        "actual_status": actual_status,
        "actual_message": actual_msg,
        "expected_status": case["expected_status"],
        "pass": passed,
        "error": error_text,
    }

results = [run_case(c) for c in all_cases]
print(f"Total cases: {len(results)}")

## 5. Ringkas Hasil Uji dalam DataFrame

In [ ]:
df_results = pd.DataFrame(results)
df_results = df_results[
    [
        "case_name",
        "input",
        "actual_status",
        "actual_message",
        "expected_status",
        "pass",
        "error",
    ]
]

# Tampilkan tabel hasil uji
print(df_results.to_string(index=False))

## 6. Visualisasi Status Lulus/Gagal

In [ ]:
status_counts = df_results["pass"].value_counts().rename(index={True: "PASS", False: "FAIL"})

ax = status_counts.plot(kind="bar", color=["#2ca02c", "#d62728"], figsize=(6, 4), rot=0)
ax.set_title("Ringkasan Hasil Test Cases")
ax.set_xlabel("Status")
ax.set_ylabel("Jumlah Kasus")

for patch in ax.patches:
    ax.annotate(
        str(int(patch.get_height())),
        (patch.get_x() + patch.get_width() / 2, patch.get_height()),
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()